# Grid features — FFRK + DEM + SoilGrids на EPSG:3035

Считает фичи на регулярной 1 км сетке (EPSG:3035) study area по всем датам из
`DATES`. Использует **все** ReKIS-станции (включая holdout) как neighbour pool
для kriging-фич.

**Выход:**

- `results/bayesnf/grid_features/_grid_meta.parquet` — статические фичи (один раз):
  `grid_id, x_proj, y_proj, latitude, longitude, elevation_m, bulk_density, clay,
  sand, silt, soc, water_10kpa`
- `results/bayesnf/grid_features/<YYYY-MM-DD>.parquet` — time-varying фичи на день:
  `grid_id, datetime, idw, gos, svd_00..svd_20`

После записи всё льётся в `s3://thesis-data-ismaktam/bayesnf/grid_features/`.
Объединённый long-format фрейм по grid×day строится `JOIN` по `grid_id`, схема
совместима с fold-паркетами BayesNF / LGBM.

**Как использовать для карты:** загрузить `_grid_meta.parquet`, реконструировать
форму `(H, W)` через `pivot(index='y_proj', columns='x_proj', values=…)`, рисовать
`imshow` / `pcolormesh`.


## 1. Окружение + параметры


In [ ]:
import os, sys, json, time
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from pyproj import Transformer
from sklearn.neighbors import BallTree

# Repo root must be 3 levels up from this notebook.
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR
for _ in range(6):
    if (REPO_ROOT / 'pyproject.toml').exists() or (REPO_ROOT / 'src' / 'thesis').exists():
        break
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
print(f'repo root: {REPO_ROOT}')

from thesis.config import Config
from thesis.data.rekis import ReKISSource
from thesis.data.dem  import DEMSource
from thesis.data.soilgrids import SoilGridsSource
from thesis.grid import build_prediction_grid

import boto3
S3_BUCKET = 'thesis-data-ismaktam'
S3_PREFIX = 'bayesnf/grid_features'
LOCAL_ROOT = REPO_ROOT / 'results' / 'bayesnf' / 'grid_features'
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
print(f'local out: {LOCAL_ROOT}')

cfg = Config()
print(f'study area: lon [{cfg.study_area.lon_min}, {cfg.study_area.lon_max}]  '
      f'lat [{cfg.study_area.lat_min}, {cfg.study_area.lat_max}]  '
      f'res={cfg.study_area.grid_resolution_m} m  crs={cfg.study_area.target_crs}')


## 2. Параметры

Поменяй `DATES` под то, что нужно. Список / диапазон / одиночная дата — всё
работает. `K_NEIGHBOURS=15`, `SVD_QUANTILES=linspace(0,1,21)` — те же дефолты,
что в FFRK-пайплайне для станций (`src/thesis/scripts/run_grk_kfold_cv.py`),
чтобы фичи были совместимы по семантике.

**Объём данных:** при 1 км сетке ≈ 280 000 ячеек × ~40 КБ/день. 1 день ≈ 50 МБ,
≈ 30–60 с компьюта (BallTree + 280 K запросов k=15). Для одной карты — мгновенно,
для 23 К дней — сутки. Параллелизуй на vast по диапазону дат.


In [ ]:
# ─── Geometry ─────────────────────────────────────────────────────────
GRID_RESOLUTION_M = cfg.study_area.grid_resolution_m   # default 1000 (1 km)

# ─── Dates: configure as needed ───────────────────────────────────────
# Single day:
DATES = ['2018-07-15']
# Multiple days:
# DATES = ['2018-07-15', '2018-07-16', '2018-07-17']
# Date range:
# DATES = pd.date_range('2018-07-01', '2018-07-31', freq='D').strftime('%Y-%m-%d').tolist()

# ─── FFRK ─────────────────────────────────────────────────────────────
K_NEIGHBOURS  = 15
SVD_QUANTILES = np.linspace(0.0, 1.0, 21)        # 21 уровней → svd_00..svd_20
assert len(SVD_QUANTILES) == 21

# ─── Source: which stations form the neighbour pool ──────────────────
# Для финальной карты — все станции (train pool + holdout).
EXCLUDE_HOLDOUT = False

# ─── Upload toggle ────────────────────────────────────────────────────
UPLOAD_TO_S3 = True

print(f'dates: {len(DATES)} → first={DATES[0]} last={DATES[-1]}')
print(f'k_neighbours={K_NEIGHBOURS}  svd quantiles={len(SVD_QUANTILES)}')
print(f'exclude_holdout={EXCLUDE_HOLDOUT}  upload={UPLOAD_TO_S3}')


## 3. Grid + статические фичи

`_grid_meta.parquet` собирается **один раз**: координаты ячейки + DEM elevation
+ depth-averaged soil переменные. Перезаписывается, если уже существует.


In [ ]:
print('[3.1] building prediction grid')
dem = DEMSource(cfg)
pg  = build_prediction_grid(cfg, dem=dem)
H, W = pg.shape
n_cells = H * W
print(f'  grid shape: {H} × {W} = {n_cells:,} ячеек  res={GRID_RESOLUTION_M} m')

x_proj = pg.coords_proj[:, 0].astype(np.float64)
y_proj = pg.coords_proj[:, 1].astype(np.float64)

# Сразу проецируем в WGS84 для удобства рисования карт
t_inv = Transformer.from_crs(cfg.study_area.target_crs, 'EPSG:4326', always_xy=True)
lons, lats = t_inv.transform(x_proj, y_proj)

elev = pg.elevation_m.astype(np.float64) if pg.elevation_m is not None else np.zeros(n_cells)
print(f'  elev: min={elev.min():.0f}  max={elev.max():.0f}  mean={elev.mean():.0f}')

print('[3.2] sampling SoilGrids (6 vars, depth-averaged)')
SOIL_VARS = ('bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa')
soil_arrays = {}
for var in SOIL_VARS:
    src = SoilGridsSource(cfg, variable=var)   # depth=None → depth-average
    vals = src.sample_at_projected(x_proj, y_proj).astype(np.float32)
    soil_arrays[var] = vals
    print(f'  {var:14s} mean={vals.mean():.2f}  nan={np.isnan(vals).sum()}')

grid_id = np.arange(n_cells, dtype=np.int64)
meta_df = pd.DataFrame({
    'grid_id':     grid_id,
    'x_proj':      x_proj,
    'y_proj':      y_proj,
    'latitude':    lats.astype(np.float64),
    'longitude':   lons.astype(np.float64),
    'elevation_m': elev,
    **{v: soil_arrays[v] for v in SOIL_VARS},
})
print(f'  meta_df: {meta_df.shape}')
meta_df.head(3)


In [ ]:
# save + upload static meta
meta_path = LOCAL_ROOT / '_grid_meta.parquet'
meta_df.to_parquet(meta_path, index=False)
sz = meta_path.stat().st_size / 1e6
print(f'saved {meta_path}  ({sz:.1f} MB)')

if UPLOAD_TO_S3:
    s3 = boto3.client('s3', region_name='eu-north-1')
    s3.upload_file(str(meta_path), S3_BUCKET, f'{S3_PREFIX}/_grid_meta.parquet')
    print(f'uploaded → s3://{S3_BUCKET}/{S3_PREFIX}/_grid_meta.parquet')


## 4. Станции (neighbour pool)

Грузим все ReKIS-станции с rainfall для требуемых дат. Проецируем в EPSG:3035.
Для каждого `date` отделяем wet-only подмножество, чтобы FFRK работал по тем же
правилам, что в обучающих фолд-паркетах (там svd-квантили считались от
neighbour pool на дне).


In [ ]:
print('[4.1] loading ReKIS rainfall for date range')
rekis = ReKISSource(cfg)
d_lo, d_hi = min(DATES), max(DATES)
all_rain = rekis.load(d_lo, d_hi, exclude_holdout=EXCLUDE_HOLDOUT)
print(f'  rows: {len(all_rain):,}  stations: {all_rain["station_id"].nunique()}  '
      f'dates: {all_rain["date"].nunique()}')

print('[4.2] projecting stations to EPSG:3035')
t_proj = Transformer.from_crs('EPSG:4326', cfg.study_area.target_crs, always_xy=True)
# unique-per-station table
stn_meta = (all_rain[['station_id', 'lon', 'lat']]
            .drop_duplicates('station_id').reset_index(drop=True))
stn_meta['x_proj'], stn_meta['y_proj'] = t_proj.transform(
    stn_meta['lon'].values, stn_meta['lat'].values)
print(f'  stations projected: {len(stn_meta)}')

# Pre-build station→xy lookup
sid_to_xy = dict(zip(stn_meta['station_id'],
                     zip(stn_meta['x_proj'], stn_meta['y_proj'])))

# Wet-station precip per day (long table). For consistency with the fold pipeline
# we DO include zero-precipitation stations in the neighbour pool — FFRK on grid
# cells operates on all available rainfall values of the day.
all_rain = all_rain.dropna(subset=['precip_mm']).reset_index(drop=True)
print(f'  rows after NaN drop: {len(all_rain):,}')


## 5. FFRK для grid-ячеек

Адаптируем `compute_day_geo_features` из `thesis.models.grk.features`:
вместо запросов по станциям — запросы по ячейкам грида, при том что
**neighbour pool — всегда все станции дня**. Возвращаем `(n_cells, 23)` массив
с колонками `idw, gos, svd_00..svd_20`.


In [ ]:
def grid_ffrk_one_day(
    date: str,
    grid_xy: np.ndarray,          # (n_cells, 2) EPSG:3035 — query points
    stn_xy: np.ndarray,           # (n_stn, 2) EPSG:3035 — neighbour pool
    stn_z:  np.ndarray,           # (n_stn,) precip_mm
    k: int = K_NEIGHBOURS,
    svd_quantiles: np.ndarray = SVD_QUANTILES,
) -> pd.DataFrame:
    n_cells = grid_xy.shape[0]
    n_stn   = stn_xy.shape[0]
    n_q     = len(svd_quantiles)

    if n_stn < 2:
        # Bail out: not enough neighbours.
        empty = np.zeros((n_cells, 2 + n_q), dtype=np.float32)
        cols  = ['idw', 'gos'] + [f'svd_{i:02d}' for i in range(n_q)]
        return pd.DataFrame(empty, columns=cols)

    k_eff = min(k, n_stn)
    tree = BallTree(stn_xy, metric='euclidean')
    dists, idxs = tree.query(grid_xy, k=k_eff)   # (n_cells, k_eff)

    z_nbr = stn_z[idxs]                          # (n_cells, k_eff)
    d_nbr = dists.astype(np.float64)

    # IDW: 1/d² weights → weighted mean
    w = 1.0 / (d_nbr ** 2 + 1e-8)
    w /= w.sum(axis=1, keepdims=True)
    idw = (w * z_nbr).sum(axis=1).astype(np.float32)

    # SVD: per-cell quantiles of neighbour precip
    svd = np.quantile(z_nbr, svd_quantiles, axis=1).T.astype(np.float32)
    # ↑ shape (n_cells, n_q)

    # GOS — gaussian similarity between each neighbour's "profile" and the cell-
    # level profile. The profile of a single neighbour is just `(z, z, …)` →
    # gaussian-of-similarity collapses to a sum of exp(-(z_nbr_i − svd_local)²).
    sigma = np.ones(n_q, dtype=np.float64) + 1e-8
    profile = svd.astype(np.float64)                            # (n_cells, n_q)
    # neighbour profile = repeat z along quantile dim
    nbr_profiles = np.broadcast_to(
        z_nbr[..., None], (n_cells, k_eff, n_q)).astype(np.float64)
    diff = (nbr_profiles - profile[:, None, :]) / sigma
    S = np.exp(-(diff ** 2).sum(axis=-1))                       # (n_cells, k_eff)
    S_sum = S.sum(axis=1, keepdims=True) + 1e-8
    gos = (S / S_sum * z_nbr).sum(axis=1).astype(np.float32)

    cols = ['idw', 'gos'] + [f'svd_{i:02d}' for i in range(n_q)]
    arr  = np.column_stack([idw, gos, svd])
    return pd.DataFrame(arr, columns=cols)


# Sanity check: tiny test
_test = grid_ffrk_one_day(
    date='_test',
    grid_xy=np.array([[4_400_000, 3_100_000], [4_500_000, 3_200_000]], dtype=np.float64),
    stn_xy=stn_meta[['x_proj', 'y_proj']].values[:50],
    stn_z=np.random.RandomState(0).rand(50) * 10,
)
print('helper output shape:', _test.shape)
print(_test.head())


## 6. Главный цикл — день за днём

Для каждой даты:
1. Селектим станции дня и их precip_mm.
2. Считаем FFRK для всех `n_cells` ячеек.
3. Складываем `(grid_id, datetime, idw, gos, svd_*)` → parquet → S3.

Файл per-date — это лёгкий time-varying слой, статика лежит отдельно в
`_grid_meta.parquet`. Long-format join восстанавливается одним merge'ом.


In [ ]:
s3 = boto3.client('s3', region_name='eu-north-1') if UPLOAD_TO_S3 else None
grid_xy = np.column_stack([x_proj, y_proj])
grid_id_arr = np.arange(n_cells, dtype=np.int64)

stations_by_date = dict(tuple(all_rain.groupby('date')))

for di, date in enumerate(DATES, 1):
    t0 = time.time()
    day_df = stations_by_date.get(date)
    if day_df is None or len(day_df) < 2:
        print(f'[{di}/{len(DATES)}] {date}: SKIP — нет станций с rainfall')
        continue

    stn_xy = day_df[['station_id']].merge(stn_meta, on='station_id')[
        ['x_proj', 'y_proj']].values
    # ↑ keep station-id ↔ xy order alignment with z
    z = day_df['precip_mm'].values.astype(np.float32)

    feats = grid_ffrk_one_day(date, grid_xy, stn_xy, z)
    feats.insert(0, 'datetime', pd.to_datetime(date))
    feats.insert(0, 'grid_id', grid_id_arr)

    out_path = LOCAL_ROOT / f'{date}.parquet'
    feats.to_parquet(out_path, index=False)
    sz = out_path.stat().st_size / 1e6

    if UPLOAD_TO_S3:
        s3.upload_file(str(out_path), S3_BUCKET, f'{S3_PREFIX}/{date}.parquet')

    dt = time.time() - t0
    print(f'[{di}/{len(DATES)}] {date}: n_stn={len(z):4d}  '
          f'feats={feats.shape}  {sz:.1f} MB  in {dt:.1f}s  '
          f'{"→ S3" if UPLOAD_TO_S3 else ""}')

print('done')


## 7. Карта — quick preview

Чтобы построить карту на любую дату:

1. Грузим `_grid_meta.parquet` и `<date>.parquet`
2. Merge по `grid_id`
3. Reshape в `(H, W)` через `pivot` — для отрисовки

Дальше уже всё что угодно: `imshow` с `extent=`, `pcolormesh` на проекции,
overlay со станциями. Маска NaN там, где `elevation_m` за границей DEM.


In [ ]:
import matplotlib.pyplot as plt

PREVIEW_DATE = DATES[0]
PREVIEW_FEATURE = 'idw'   # один из: idw, gos, svd_00..svd_20, elevation_m, *_soil

meta = pd.read_parquet(LOCAL_ROOT / '_grid_meta.parquet')
day  = pd.read_parquet(LOCAL_ROOT / f'{PREVIEW_DATE}.parquet')
joined = meta.merge(day, on='grid_id', how='left')
print('joined columns:', joined.columns.tolist())

# Reconstruct (H, W) raster — coords aligned with build_prediction_grid output
H_, W_ = H, W
img = joined[PREVIEW_FEATURE].values.reshape(H_, W_)

xmin = joined['x_proj'].min(); xmax = joined['x_proj'].max()
ymin = joined['y_proj'].min(); ymax = joined['y_proj'].max()

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
im = ax.imshow(img, origin='lower', extent=[xmin, xmax, ymin, ymax],
               cmap='viridis', aspect='equal')
fig.colorbar(im, ax=ax, label=PREVIEW_FEATURE)
ax.set_title(f'{PREVIEW_FEATURE} @ {PREVIEW_DATE}  (EPSG:{cfg.study_area.target_crs.split(":")[-1]})')
ax.set_xlabel('x_proj (m)'); ax.set_ylabel('y_proj (m)')
plt.tight_layout()
plt.show()


## 8. Использование выхода

**Для LGBM:** загрузи `_grid_meta.parquet` + любое количество `<date>.parquet`,
сделай inner-join по `grid_id`, передай в `lgbm.Booster.predict()` те же
feature_cols, что были на обучении.

**Для BayesNF:** на predict добавь `datetime` (timestamp) и предсказывай
покольцово по чанкам ячеек (см. `predict_bnf` JIT-patch в `baseline.ipynb`).

**Для карты:** см. cell выше — `pivot` по `(y_proj, x_proj)` или
`.values.reshape(H, W)` сразу даёт raster.

**Параллельный запуск на Vast:** разбей `DATES` на N кусков, запусти N копий
ноутбука (каждой свой `DATES[start:end]`), они независимо пишут в S3 — никаких
коллизий по ключу.
